In [25]:
# Import necessary libraries
import pandas as pd

In [26]:
# Load the raw dataset from the data directory
df = pd.read_csv('../data/raw/US houuse price of 10 states.csv')

In [27]:
df.head(20)

,date,house_size,bed,bath,price,broker,street,city,state_name,zip_code
0,"AUG 29, 2024","2,520 sqft (on 0.44 acres)",4bd,4bd,"$415,000",NaN,2004 W 23rd Ct,Panama City,Florida,32405
1,"AUG 29, 2024",704 sqft (on 0.33 acres),2bd,2bd,"$58,000",NaN,5390 Webb St,Graceville,Florida,32440
2,"AUG 29, 2024","1,926 sqft (on 0.45 acres)",3bd,3bd,"$375,000",Coldwell Banker Hartung,6761 Landover Cir,Tallahassee,Florida,32317
3,"AUG 29, 2024","1,132 sqft",Studio,Studio,"$190,000","EXP Realty, LLC",1701 S Fairfield Dr,Perdido Key,Florida,32507
4,"AUG 29, 2024","1,205 sqft",3bd,3bd,"$233,900","D R Horton Realty of NW Florida, LLC",6274 June Bug Dr,Milton,Florida,32583
5,"AUG 29, 2024","3,044 sqft (on 0.34 acres)",4bd,4bd,"$416,402","ADAMS HOME REALTY, INC",6528 Benelli Dr,Milton,Florida,32570
6,"AUG 29, 2024",NaN,NaN,NaN,"$30,000",Waypoint Properties,Lot 34&35 Menomini St #3,Crawfordville,Florida,32327
7,"AUG 29, 2024",NaN,NaN,NaN,"$2,650,000",Compass,Ac29 Sea Garden St,Alys Beach,Florida,32461
8,"AUG 28, 2024","1,422 sqft",2bd,2bd,"$455,000",NaN,122 3rd St,Mexico Beach,Florida,32456
9,"AUG 28, 2024","1,192 sqft",2bd,2bd,"$325,000",NaN,4026 Kirkpatrick Rd,Southport,Florida,32409


In [28]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12075 entries, 0 to 12074
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        12075 non-null  str  
 1   house_size  11107 non-null  str  
 2   bed         11410 non-null  str  
 3   bath        11410 non-null  str  
 4   price       11009 non-null  str  
 5   broker      9569 non-null   str  
 6   street      12075 non-null  str  
 7   city        12075 non-null  str  
 8   state_name  12075 non-null  str  
 9   zip_code    12075 non-null  int64
dtypes: int64(1), str(9)
memory usage: 943.5 KB


## Data Cleaning Steps

### 1. Convert `date` to datetime

In [29]:
df['date'] = pd.to_datetime(df['date'])
display(df['date'].head())

/var/folders/20/hjsmdy4x7mb2m4mdsq9ml3lm0000gn/T/ipykernel_38969/1293998612.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'])


0   2024-08-29
1   2024-08-29
2   2024-08-29
3   2024-08-29
4   2024-08-29
Name: date, dtype: datetime64[us]

### 2. Extract `house_size` and `lot_acres`

In [30]:
import re

def extract_house_and_lot_size(house_size_str):
    house_sqft = None
    lot_acres = None

    if pd.isna(house_size_str):
        return None, None

    # Extract house_sqft
    sqft_match = re.search(r'([\d,]+)\s*sqft', house_size_str)
    if sqft_match:
        house_sqft = float(sqft_match.group(1).replace(',', ''))

    # Extract lot_acres
    acres_match = re.search(r'\((?:on\s)?([\d.]+)\s*acres\)', house_size_str)
    if acres_match:
        lot_acres = float(acres_match.group(1))

    return house_sqft, lot_acres

df[['house_sqft', 'lot_acres']] = df['house_size'].apply(lambda x: pd.Series(extract_house_and_lot_size(x)))

display(df[['house_size', 'house_sqft', 'lot_acres']].head())

,house_size,house_sqft,lot_acres
0,"2,520 sqft (on 0.44 acres)",2520.0,0.44
1,704 sqft (on 0.33 acres),704.0,0.33
2,"1,926 sqft (on 0.45 acres)",1926.0,0.45
3,"1,132 sqft",1132.0,NaN
4,"1,205 sqft",1205.0,NaN


### 3. Clean and convert `bed` and `bath` to numeric

In [31]:
def clean_bed_bath(value):
    if pd.isna(value):
        return None
    if 'Studio' in str(value):
        return 0  # Assuming studio means 0 separate bedrooms/bathrooms
    if 'bd' in str(value):
        return float(str(value).replace('bd', '').strip())
    return None

df['bed'] = df['bed'].apply(clean_bed_bath)
df['bath'] = df['bath'].apply(clean_bed_bath)

display(df[['bed', 'bath']].head())

,bed,bath
0,4.0,4.0
1,2.0,2.0
2,3.0,3.0
3,0.0,0.0
4,3.0,3.0


### 4. Clean and convert `price` to numeric

In [32]:
df['price'] = df['price'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

display(df['price'].head())

0    415000.0
1     58000.0
2    375000.0
3    190000.0
4    233900.0
Name: price, dtype: float64

## 6. Standardize `state_name` column

In [33]:
unique_states = df['state_name'].unique()
print("Unique States:")
for state in unique_states:
    print(state)

Unique States:
Florida
FL
California
Alabama
Illinois
New York
Arizona
LA
South Carolina
North Carolina
WA
CA
NY
IL
AZ
AL


In [34]:
state_name_mapping = {
    'FL': 'Florida',
    'CA': 'California',
    'IL': 'Illinois',
    'NY': 'New York',
    'AZ': 'Arizona',
    'AL': 'Alabama',
    'LA': 'Louisiana',
    'WA': 'Washington'
}
df['state_name'] = df['state_name'].replace(state_name_mapping)

print("Unique States after standardization:")
for state in df['state_name'].unique():
    print(state)

Unique States after standardization:
Florida
California
Alabama
Illinois
New York
Arizona
Louisiana
South Carolina
North Carolina
Washington


## 7. Dropping the columns with null values in size, bed and bath :

In [35]:
null_house_bed_bath_count = df[df['house_size'].isnull() & df['bed'].isnull() & df['bath'].isnull()].shape[0]
print(f"Number of rows with null values in 'house_size', 'bed', and 'bath': {null_house_bed_bath_count}")

Number of rows with null values in 'house_size', 'bed', and 'bath': 657


In [36]:
df.dropna(subset=['bed', 'bath'], inplace=True)
print(f"Number of rows after dropping nulls in 'bed' or 'bath': {df.shape[0]}")


Number of rows after dropping nulls in 'bed' or 'bath': 11410


## 8. Data Imputation

In [37]:
df.info()

<class 'pandas.DataFrame'>
Index: 11410 entries, 0 to 12074
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        11410 non-null  datetime64[us]
 1   house_size  11099 non-null  str           
 2   bed         11410 non-null  float64       
 3   bath        11410 non-null  float64       
 4   price       10444 non-null  float64       
 5   broker      9016 non-null   str           
 6   street      11410 non-null  str           
 7   city        11410 non-null  str           
 8   state_name  11410 non-null  str           
 9   zip_code    11410 non-null  int64         
 10  house_sqft  11099 non-null  float64       
 11  lot_acres   3970 non-null   float64       
dtypes: datetime64[us](1), float64(5), int64(1), str(5)
memory usage: 1.1 MB


In [38]:
df['broker'] = df['broker'].fillna('Unknown')

### Imputing price using the mean price of house with same number of beds and same state


In [39]:
df['price'] = df.groupby(['bed', 'state_name'])['price'].transform(lambda x: x.fillna(x.mean()))
display(df[['bed', 'state_name', 'price']].head())
print(f"Number of rows where 'price' is still null after grouped mean imputation: {df['price'].isnull().sum()}")

,bed,state_name,price
0,4.0,Florida,415000.0
1,2.0,Florida,58000.0
2,3.0,Florida,375000.0
3,0.0,Florida,190000.0
4,3.0,Florida,233900.0


Number of rows where 'price' is still null after grouped mean imputation: 951


### There are still some null values because some states do not have price for every house like Louisiana

In [40]:
df['price'] = df.groupby('bed')['price'].transform(lambda x: x.fillna(x.mean()))
display(df[['bed', 'state_name', 'price']].head())
print(f"Number of rows where 'price' is still null after grouped mean imputation by bed: {df['price'].isnull().sum()}")

,bed,state_name,price
0,4.0,Florida,415000.0
1,2.0,Florida,58000.0
2,3.0,Florida,375000.0
3,0.0,Florida,190000.0
4,3.0,Florida,233900.0


Number of rows where 'price' is still null after grouped mean imputation by bed: 0


### Imputung the house_sqft with mean of same number of beds and state

In [41]:
df['house_sqft'] = df.groupby(['bed', 'state_name'])['house_sqft'].transform(lambda x: x.fillna(x.mean()))
display(df[['bed', 'state_name', 'house_sqft']].head())

,bed,state_name,house_sqft
0,4.0,Florida,2520.0
1,2.0,Florida,704.0
2,3.0,Florida,1926.0
3,0.0,Florida,1132.0
4,3.0,Florida,1205.0


### There are still null value for studio apartment (0 beds)

In [42]:
null_house_sqft_rows = df[df['house_sqft'].isnull()] # dispal
display(null_house_sqft_rows[['bed', 'state_name', 'house_sqft']]) # figure out how to get size of studio apartment
print(f"Number of rows where 'house_sqft' is still null: {null_house_sqft_rows.shape[0]}")

,bed,state_name,house_sqft
8979,0.0,North Carolina,NaN
9039,0.0,North Carolina,NaN
9083,0.0,North Carolina,NaN
9114,0.0,North Carolina,NaN
9154,0.0,North Carolina,NaN
9177,0.0,North Carolina,NaN
9248,0.0,North Carolina,NaN
9250,0.0,North Carolina,NaN
9323,0.0,North Carolina,NaN


Number of rows where 'house_sqft' is still null: 9


### Impute the house_sqft for studio apartment

In [43]:
# Calculate the mean house_sqft specifically for studio apartments (bed=0)
studio_mean_house_sqft = df[df['bed'] == 0]['house_sqft'].mean()

# Impute null house_sqft values for studio apartments with this mean
df.loc[(df['bed'] == 0) & (df['house_sqft'].isnull()), 'house_sqft'] = studio_mean_house_sqft

print(f"Mean house_sqft for Studio apartments (bed=0) used for imputation: {studio_mean_house_sqft:,.2f} sqft")
print(f"Number of rows where 'house_sqft' is still null after studio-specific imputation: {df['house_sqft'].isnull().sum()}")

Mean house_sqft for Studio apartments (bed=0) used for imputation: 1,951.87 sqft
Number of rows where 'house_sqft' is still null after studio-specific imputation: 0


### Droping the lot_acres column because the missing data is around 65%


In [44]:
df.drop('lot_acres', axis=1, inplace=True)

### Dropping the house_size column because relevant data is in house_sqft

In [45]:
df.drop('house_size', axis=1, inplace=True)

In [46]:
df.info()

<class 'pandas.DataFrame'>
Index: 11410 entries, 0 to 12074
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        11410 non-null  datetime64[us]
 1   bed         11410 non-null  float64       
 2   bath        11410 non-null  float64       
 3   price       11410 non-null  float64       
 4   broker      11410 non-null  str           
 5   street      11410 non-null  str           
 6   city        11410 non-null  str           
 7   state_name  11410 non-null  str           
 8   zip_code    11410 non-null  int64         
 9   house_sqft  11410 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(1), str(4)
memory usage: 980.5 KB


In [47]:
df.head(200)

,date,bed,bath,price,broker,street,city,state_name,zip_code,house_sqft
0,2024-08-29,4.0,4.0,415000.0,Unknown,2004 W 23rd Ct,Panama City,Florida,32405,2520.0
1,2024-08-29,2.0,2.0,58000.0,Unknown,5390 Webb St,Graceville,Florida,32440,704.0
2,2024-08-29,3.0,3.0,375000.0,Coldwell Banker Hartung,6761 Landover Cir,Tallahassee,Florida,32317,1926.0
3,2024-08-29,0.0,0.0,190000.0,"EXP Realty, LLC",1701 S Fairfield Dr,Perdido Key,Florida,32507,1132.0
4,2024-08-29,3.0,3.0,233900.0,"D R Horton Realty of NW Florida, LLC",6274 June Bug Dr,Milton,Florida,32583,1205.0
...,...,...,...,...,...,...,...,...,...,...
221,2024-08-26,4.0,4.0,2625000.0,The Premier Property Group Seacrest Office,30 Clove Hitch Ln,Santa Rosa Beach,Florida,32459,3130.0
222,2024-08-26,6.0,6.0,12750000.0,Compass,8570 E County Highway 30A,Inlet Beach,Florida,32461,7650.0
223,2024-08-26,3.0,3.0,625000.0,Redfin,3711 Scenic Hwy,Pensacola,Florida,32504,3132.0
224,2024-08-26,3.0,3.0,496900.0,"Coastwise Realty, Inc.",3468 Firefly Cir,Saint Teresa,Florida,32358,1514.0


Export the cleaned data

In [48]:
import os

# Ensure the processed directory exists
output_path = "../data/processed/cleaned_house_data.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Export the cleaned dataframe
df.to_csv(output_path, index=False)

print(f"Cleaned data exported to '{output_path}'")

Cleaned data exported to '../data/processed/cleaned_house_data.csv'
